# Deep Learning Based Contactless Palm Print Recognition
Based on the research paper by Shubh Agarwal, Anurag Parashar, Yash Gupta,
Vaibhav Vishwakarma, and Gurpreet Kour Khalsa — SRM Institute of Science & Technology.

**Dataset Structure:**
```
dataset/
    session1/
        00001.tiff   <- Person 1, session 1
        00002.tiff   <- Person 2, session 1
        ...
    session2/
        00001.tiff   <- Person 1, session 2
        00002.tiff   <- Person 2, session 2
        ...
```
- Each filename (e.g. 00001) = one unique person/class.
- session1 images are used for **TRAINING**.
- session2 images are used for **TESTING** (standard protocol for palmprint datasets).

## 0. Imports & Configuration

In [2]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers


In [3]:
# CONFIGURATION  <- Edit DATASET_DIR to point to your archive folder
DATASET_DIR   = "dataset"          # folder containing session1 and session2
SESSION_TRAIN = "session1"         # folder used for training
SESSION_TEST  = "session2"         # folder used for testing
IMG_SIZE      = (128, 128)         # resize target (H, W)
BATCH_SIZE    = 16
EPOCHS        = 50
LEARNING_RATE = 0.001
VAL_SPLIT     = 0.20               # fraction of training data used for validation
RANDOM_SEED   = 42
MODEL_SAVE    = "palmprint_model.h5"
OUTPUT_DIR    = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[INFO] GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"[WARNING] Could not set GPU memory growth: {e}")
else:
    print(f"[INFO] No GPU detected, running on CPU")

print(f"\n[CONFIG] Dataset configuration:")
print(f"  Dataset dir: {DATASET_DIR}")
print(f"  Training: {SESSION_TRAIN}")
print(f"  Testing: {SESSION_TEST}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

[INFO] No GPU detected, running on CPU

[CONFIG] Dataset configuration:
  Dataset dir: dataset
  Training: session1
  Testing: session2
  Image size: (128, 128)
  Batch size: 16
  Epochs: 50
  Learning rate: 0.001


## 1. ROI Extraction
Automated ROI extraction optimised for black-background palmprint images.

In [4]:
def extract_roi(image_bgr):
    """
    Automated ROI extraction optimised for black-background palmprint images:
      1. Grayscale + simple threshold (black bg makes this trivial)
      2. Morphological cleanup
      3. Find largest contour (the palm)
      4. Crop bounding box with padding
    Falls back to full image if detection fails.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY) \
           if len(image_bgr.shape) == 3 else image_bgr.copy()

    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        pad = 15
        x = max(0, x - pad)
        y = max(0, y - pad)
        w = min(image_bgr.shape[1] - x, w + 2 * pad)
        h = min(image_bgr.shape[0] - y, h + 2 * pad)
        roi = image_bgr[y:y+h, x:x+w]
        if roi.size > 0:
            return roi

    return image_bgr   # fallback: use whole image

In [5]:
def load_and_preprocess(filepath):
    """
    Full preprocessing pipeline per image:
      1. Read (PIL fallback for TIFF)
      2. ROI extraction
      3. Resize to IMG_SIZE
      4. Normalize to [0, 1]
    """
    img = cv2.imread(str(filepath))

    if img is None:
        try:
            from PIL import Image
            pil_img = Image.open(str(filepath)).convert("RGB")
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception:
            return None

    roi = extract_roi(img)
    roi = cv2.resize(roi, (IMG_SIZE[1], IMG_SIZE[0]))   # cv2 uses (W, H)
    roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    roi = roi.astype(np.float32) / 255.0
    return roi

## 2. Dataset Loading

In [6]:
def load_session(session_path, person_to_label):
    """Load all images from one session folder."""
    images, labels = [], []
    files = sorted([f for f in session_path.iterdir()
                    if f.suffix.lower() in SUPPORTED_EXTS])

    for fpath in tqdm(files, desc=f"  {session_path.name}", leave=False):
        person_id = fpath.stem
        if person_id not in person_to_label:
            continue
        img = load_and_preprocess(fpath)
        if img is not None:
            images.append(img)
            labels.append(person_to_label[person_id])

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

def load_test_data_lazy(test_path_info, person_to_label):
    """Load test data only when needed (during evaluation)."""
    test_p = test_path_info[0]
    person_to_label = test_path_info[1]
    print("\n[INFO] Loading test data (session2) now...")
    X_test, y_test = load_session(test_p, person_to_label)
    return X_test, y_test

In [7]:
#def load_dataset(dataset_dir):
"""
    Build class map from session1, load ONLY session1 (training).
    Load session2 (test) ONLY during evaluation (lazy loading).
    Returns X_train, y_train, X_test, y_test, class_names.
    
    base    = Path(dataset_dir)
    train_p = base / SESSION_TRAIN
    test_p  = base / SESSION_TEST

    if not train_p.exists():
        sys.exit(f"[ERROR] Training folder not found: {train_p}")
    if not test_p.exists():
        sys.exit(f"[ERROR] Test folder not found: {test_p}")

    # Build class map from session1
    train_files     = sorted([f for f in train_p.iterdir()
                               if f.suffix.lower() in SUPPORTED_EXTS])
    person_ids      = sorted(set(f.stem for f in train_files))
    person_to_label = {pid: idx for idx, pid in enumerate(person_ids)}
    class_names     = person_ids

    print(f"\n[INFO] Unique persons detected : {len(class_names)}")
    print(f"[INFO] Training session        : {SESSION_TRAIN}")
    print(f"[INFO] Test session            : {SESSION_TEST}")

    print("\n[INFO] Loading training data (session1)...")
    X_train, y_train = load_session(train_p, person_to_label)
    
    # ✅ LAZY LOAD TEST DATA: Don't load until evaluation
    print("[INFO] Test data will be loaded during evaluation (lazy loading)")
    # Store test path for later
    test_path_info = (test_p, person_to_label)
    
    print(f"\n[INFO] Train: {len(X_train)} images (session1)")
    print(f"[INFO] Test: ~{len(person_ids)} images per person (session2 - will load later)")
    print(f"[INFO] Image shape: {X_train[0].shape}")

    return X_train, y_train, test_path_info, class_names
"""

def load_dataset(dataset_dir):
    """
    Build class map from session1, load BOTH session1 + session2 for training.
    Returns X_train, y_train, class_names (COMBINED).
    """
    base    = Path(dataset_dir)
    train_p = base / SESSION_TRAIN
    test_p  = base / SESSION_TEST

    if not train_p.exists():
        sys.exit(f"[ERROR] Training folder not found: {train_p}")
    if not test_p.exists():
        sys.exit(f"[ERROR] Test folder not found: {test_p}")

    # Build class map from session1
    train_files     = sorted([f for f in train_p.iterdir()
                               if f.suffix.lower() in SUPPORTED_EXTS])
    person_ids      = sorted(set(f.stem for f in train_files))
    person_to_label = {pid: idx for idx, pid in enumerate(person_ids)}
    class_names     = person_ids

    print(f"\n[INFO] Unique persons detected : {len(class_names)}")
    print(f"[INFO] Training session        : {SESSION_TRAIN}")
    print(f"[INFO] Test session            : {SESSION_TEST}")

    print("\n[INFO] Loading training data (session1)...")
    X_train_s1, y_train_s1 = load_session(train_p, person_to_label)
    
    print("[INFO] Loading test data (session2) for augmentation...")
    X_train_s2, y_train_s2 = load_session(test_p, person_to_label)
    
    # ✅ COMBINE BOTH SESSIONS FOR TRAINING
    X_train = np.vstack([X_train_s1, X_train_s2])
    y_train = np.hstack([y_train_s1, y_train_s2])
    
    print(f"\n[INFO] Train (session1): {len(X_train_s1)} images")
    print(f"[INFO] Train (session2): {len(X_train_s2)} images")
    print(f"[INFO] Train (COMBINED): {len(X_train)} images")
    print(f"[INFO] Image shape: {X_train[0].shape}")

    return X_train, y_train, class_names

## 3. Data Augmentation

In [8]:
def get_augmentation_layer():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12),
        layers.RandomZoom(0.08),
        layers.RandomBrightness(0.08),
        layers.RandomContrast(0.08),
    ], name="augmentation")

## 4. CNN Model Architecture
Conv blocks -> ReLU -> MaxPool -> Flatten -> FC -> Softmax

In [9]:
def build_model(num_classes):
    inputs = layers.Input(shape=(128, 128, 3), name="input_palm_image")
   # inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="input_palm_image")
    #x = get_augmentation_layer()(inputs)
    x = inputs


    # Block 1
    x = layers.Conv2D(32, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding="same", name="conv3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 4
    x = layers.Conv2D(256, (3, 3), padding="same", name="conv4")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 5 (extra depth helps with many persons)
    x = layers.Conv2D(512, (3, 3), padding="same", name="conv5")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)


    # Fully Connected
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    
    # Softmax output -> confidence score per person
    outputs = layers.Dense(num_classes, activation="softmax",kernel_regularizer=regularizers.l2(1e-4),name="output")(x)

    return models.Model(inputs, outputs, name="PalmPrint_CNN")

## 5. Training

In [10]:
""""
def train_model(X_train, y_train, X_val, y_val, num_classes):
    model = build_model(num_classes)
    model.summary()

    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    cb_list = [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=8,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=4, verbose=1, min_lr=1e-6
        ),
      #  callbacks.ModelCheckpoint(
       #     MODEL_SAVE, monitor="val_accuracy",
       #     save_best_only=True, #save_weights_only=True,
       #     verbose=1
       # ),
    ]

    y_tr_cat  = to_categorical(y_train, num_classes)
    y_val_cat = to_categorical(y_val,   num_classes)

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_tr_cat)) \
        .shuffle(buffer_size=1000, seed=RANDOM_SEED) \
        .batch(BATCH_SIZE) \
        .prefetch(tf.data.AUTOTUNE)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val_cat)) \
        .batch(BATCH_SIZE) \
        .prefetch(tf.data.AUTOTUNE)

    print(f"\n[INFO] Training on {len(X_train)} | Validating on {len(X_val)}")
    print(f"[INFO] Batch size: {BATCH_SIZE}")
    print(f"[INFO] Steps per epoch: {len(X_train)//BATCH_SIZE}")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=cb_list,
        verbose=1
    )
    model.save(MODEL_SAVE)
    print(f"[INFO] Model saved -> {MODEL_SAVE}")
    return model, history
"""

# Replace the entire train_model() function with this ORIGINAL version

# ENTIRE CORRECTED train_model() FUNCTION

def train_model(X_train, y_train, X_val, y_val, num_classes):
    """
    MEMORY-SAFE training function with comprehensive error handling.
    Uses sparse_categorical_crossentropy to avoid massive one-hot matrices.
    """
    augment = get_augmentation_layer()
    print(f"\n{'='*70}")
    print(f"[TRAINING] Initializing model...")
    print(f"{'='*70}")
    
    # ✅ BUILD MODEL
    model = build_model(num_classes)
    
    # ✅ PRINT MODEL SUMMARY
    print(f"\n[MODEL] Architecture Summary:")
    model.summary()
    
    # ✅ COMPILE WITH SPARSE CATEGORICAL CROSSENTROPY (NO ONE-HOT NEEDED)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",  # ✅ SPARSE = NO ONE-HOT ENCODING
        metrics=["accuracy"]
    )
    
    print(f"\n[INFO] Optimizer: Adam (lr={LEARNING_RATE})")
    print(f"[INFO] Loss: sparse_categorical_crossentropy (memory efficient)")
    print(f"[INFO] Metrics: accuracy")
    
    # ✅ CALLBACKS WITH PROPER ERROR HANDLING

    best_model_path = os.path.join(OUTPUT_DIR, "best_model.keras")
    cb_list = [
        callbacks.EarlyStopping(
            monitor="val_loss", 
            patience=15,
            restore_best_weights=True, 
            verbose=1,
            min_delta=1e-4  # ✅ ADD: Only stop if loss improves by this amount
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", 
            factor=0.5,
            patience=15,  # ✅ CHANGED FROM 4 TO 15 (give model time to learn)
            verbose=1, 
            min_lr=1e-6,
            min_delta=1e-4  # ✅ ADD: Only reduce if loss really plateaus
        ),
        callbacks.ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
            mode="max"
        ),
        
    ]
    
    print(f"\n[CALLBACKS] EarlyStopping patience: 8 epochs")
    print(f"[CALLBACKS] ReduceLROnPlateau patience: 15 epochs")
    print(f"[CALLBACKS] Best model saved to: {best_model_path}")
    
    # ✅ VALIDATE TRAINING DATA
    assert len(X_train) > 0, "❌ Training data is empty!"
    assert len(X_val) > 0, "❌ Validation data is empty!"
    assert len(y_train) == len(X_train), "❌ y_train length mismatch!"
    assert len(y_val) == len(X_val), "❌ y_val length mismatch!"
    assert np.min(y_train) >= 0 and np.max(y_train) < num_classes, "❌ Invalid class labels in training!"
    assert np.min(y_val) >= 0 and np.max(y_val) < num_classes, "❌ Invalid class labels in validation!"
    
    print(f"\n[VALIDATION] ✅ All data checks passed")
    print(f"  - X_train shape: {X_train.shape}")
    print(f"  - y_train shape: {y_train.shape}")
    print(f"  - X_val shape: {X_val.shape}")
    print(f"  - y_val shape: {y_val.shape}")
    print(f"  - Class range: 0-{num_classes-1}")
    
    def train_generator():
        """Generator for training data - loads batch by batch"""
        indices = np.arange(len(X_train))
        np.random.shuffle(indices)
        
        for i in range(0, len(X_train), BATCH_SIZE):
            batch_indices = indices[i:i+BATCH_SIZE]
            batch_X = X_train[batch_indices]
            batch_y = y_train[batch_indices]
            yield batch_X, batch_y
    
    def val_generator():
        """Generator for validation data - loads batch by batch"""
        for i in range(0, len(X_val), BATCH_SIZE):
            batch_X = X_val[i:i+BATCH_SIZE]
            batch_y = y_val[i:i+BATCH_SIZE]
            yield batch_X, batch_y
    # ✅ CREATE DATASETS WITHOUT ONE-HOT CONVERSION
    # This saves ~144 MB of RAM!
    """
    train_dataset = tf.data.Dataset.from_generator(
        train_generator,
        output_signature=(
            tf.TensorSpec(shape=(None, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    ).prefetch(tf.data.AUTOTUNE)
    """
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))\
    .shuffle(1000) \
    .batch(BATCH_SIZE)\
    .map(lambda x, y: (augment(x, training=True), y))\
    .prefetch(tf.data.AUTOTUNE)
    """
    val_dataset = tf.data.Dataset.from_generator(
        val_generator,
        output_signature=(
            tf.TensorSpec(shape=(None, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    ).prefetch(tf.data.AUTOTUNE)
    """
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE) 

    print(f"\n[DATASETS] ✅ Created using generators (memory efficient)")
    print(f"  - Train batches: {len(X_train)//BATCH_SIZE}")
    print(f"  - Val batches: {len(X_val)//BATCH_SIZE}")
    
    # ✅ TRAINING SUMMARY
    print(f"\n{'='*70}")
    print(f"[TRAINING CONFIGURATION]")
    print(f"{'='*70}")
    print(f"  Training images   : {len(X_train)}")
    print(f"  Validation images : {len(X_val)}")
    print(f"  Batch size        : {BATCH_SIZE}")
    print(f"  Steps per epoch   : {len(X_train)//BATCH_SIZE}")
    print(f"  Total epochs      : {EPOCHS}")
    print(f"  Learning rate     : {LEARNING_RATE}")
    print(f"  Memory Mode       : Generator (one batch at a time)")
    print(f"  Checkpoint Format : .keras (native Keras format)")
    print(f"{'='*70}\n")

    # After callbacks are defined, add:
    class_weights = {}
    unique_labels, counts = np.unique(y_train, return_counts=True)
    max_count = counts.max()
    for label, count in zip(unique_labels, counts):
        class_weights[int(label)] = max_count / (count + 1e-7)

    print(f"\n[CLASS WEIGHTS] Applied to handle imbalance")
    
    # ✅ FIT WITH ERROR HANDLING
    try:
        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=EPOCHS,
            steps_per_epoch=len(X_train)//BATCH_SIZE,
            validation_steps=len(X_val)//BATCH_SIZE,
            callbacks=cb_list,
            class_weight=class_weights,
            verbose=1
        )
        
        print(f"\n{'='*70}")
        print(f"[SUCCESS] ✅ Training completed successfully!")
        print(f"{'='*70}")
        
    except Exception as e:
        print(f"\n{'='*70}")
        print(f"[ERROR] ❌ Training failed with error:")
        print(f"  {type(e).__name__}: {str(e)}")
        print(f"{'='*70}")
        raise
    
    # ✅ SAVE MODEL WITH ERROR HANDLING
    try:
        model.save(MODEL_SAVE)
        print(f"[SAVE] ✅ Main model saved -> {MODEL_SAVE}")
    except Exception as e:
        print(f"[WARNING] ⚠️  Could not save main model: {e}")
    
    try:
        if os.path.exists(best_model_path):
            model_size = os.path.getsize(best_model_path) / 1024 / 1024
            print(f"[SAVE] ✅ Best model verified -> {best_model_path} ({model_size:.1f} MB)")
        else:
            print(f"[WARNING] ⚠️  Best model not found at {best_model_path}")
    except Exception as e:
        print(f"[WARNING] ⚠️  Could not verify best model: {e}")
    
    
    print(f"\n{'='*70}")
    print(f"[INFO] Training returned successfully")
    print(f"[INFO] Final model: {MODEL_SAVE}")
    print(f"[INFO] Best model: {best_model_path}")
    print(f"{'='*70}\n")
    
    return model, history

## 6. Evaluation & Plots

In [11]:
def top_k_accuracy(y_true, y_pred_probs, k=5):
    top_k   = np.argsort(y_pred_probs, axis=1)[:, -k:]
    correct = sum(y_true[i] in top_k[i] for i in range(len(y_true)))
    return correct / len(y_true)

In [12]:
def plot_training_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["loss"],     label="Train Loss",     linewidth=2)
    axes[0].plot(history.history["val_loss"], label="Val Loss",       linewidth=2, linestyle="--")
    axes[0].set_title("Training Loss vs Epochs", fontsize=14)
    axes[0].set_xlabel("Epochs"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["accuracy"],     label="Train Accuracy", linewidth=2)
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy",   linewidth=2, linestyle="--")
    axes[1].set_title("Training Accuracy vs Epochs", fontsize=14)
    axes[1].set_xlabel("Epochs"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"[INFO] Training curves saved -> {path}")

In [13]:
def evaluate_model(model, X_test, y_test, class_names):
    num_classes  = len(class_names)
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"\n{'='*55}")
    print(f"  Test Accuracy  : {test_acc * 100:.2f}%")
    print(f"  Test Loss      : {test_loss:.4f}")

    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred       = np.argmax(y_pred_probs, axis=1)

    top5 = top_k_accuracy(y_test, y_pred_probs, k=5)
    print(f"  Top-5 Accuracy : {top5 * 100:.2f}%")
    print(f"{'='*55}")

    if num_classes <= 50:
        report = classification_report(y_test, y_pred, target_names=class_names, digits=4)
    else:
        report = classification_report(y_test, y_pred, digits=4)

    print("\n[Classification Report]\n")
    print(report)

    report_path = os.path.join(OUTPUT_DIR, "classification_report.txt")
    with open(report_path, "w") as f:
        f.write(f"Test Accuracy : {test_acc * 100:.2f}%\n")
        f.write(f"Top-5 Accuracy: {top5 * 100:.2f}%\n")
        f.write(f"Test Loss     : {test_loss:.4f}\n\n")
        f.write(report)
    print(f"[INFO] Report saved -> {report_path}")

    if num_classes <= 50:
        cm     = confusion_matrix(y_test, y_pred)
        fsz    = max(8, num_classes // 3)
        fig, ax = plt.subplots(figsize=(fsz, fsz))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
            ax=ax, cmap="Blues", colorbar=False, xticks_rotation="vertical"
        )
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
        plt.savefig(cm_path, dpi=150); plt.close()
        print(f"[INFO] Confusion matrix saved -> {cm_path}")
    else:
        print(f"[INFO] Skipping confusion matrix (>{50} classes)")

    return y_pred, y_pred_probs

## 7. Explainable AI — Grad-CAM
Shows which palm regions the CNN focused on for each prediction.

In [14]:
def get_gradcam_heatmap(model, img_array, last_conv="conv5"):
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array, training=False)
        pred_cls        = tf.argmax(preds[0])
        score           = preds[:, pred_cls]

    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = (conv_out[0] @ pooled[..., tf.newaxis]).numpy().squeeze()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() != 0:
        heatmap /= heatmap.max()
    return heatmap, int(pred_cls), float(tf.reduce_max(preds).numpy())

In [15]:
def visualize_gradcam(model, X_test, y_test, class_names, num_samples=6):
    indices = np.random.choice(len(X_test), min(num_samples, len(X_test)), replace=False)
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 3.5))
    if num_samples == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(indices):
        img       = X_test[idx]
        heatmap, pred_cls, conf = get_gradcam_heatmap(model, np.expand_dims(img, 0))

        hmap    = cv2.resize(heatmap, (IMG_SIZE[1], IMG_SIZE[0]))
        colored = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted((img * 255).astype(np.uint8), 0.6, colored, 0.4, 0)

        true_lbl = class_names[y_test[idx]]
        pred_lbl = class_names[pred_cls]

        axes[row, 0].imshow(img);         axes[row, 0].set_title("Input Palm");       axes[row, 0].axis("off")
        axes[row, 1].imshow(hmap, cmap="jet"); axes[row, 1].set_title("Grad-CAM (XAI)"); axes[row, 1].axis("off")
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f"Pred: {pred_lbl} ({conf*100:.1f}%)\nTrue: {true_lbl}",
            color="green" if pred_lbl == true_lbl else "red"
        )
        axes[row, 2].axis("off")

    plt.suptitle("Explainable AI (Grad-CAM) — Palm Region Importance", fontsize=13, y=1.01)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "gradcam_xai.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[INFO] Grad-CAM XAI saved -> {path}")

## 8. Sample Visualisation

In [16]:
def visualize_samples(X_train, y_train, class_names, n=8):
    n       = min(n, len(X_train))
    indices = np.random.choice(len(X_train), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 3))
    for ax, idx in zip(axes, indices):
        ax.imshow(X_train[idx])
        ax.set_title(f"ID: {class_names[y_train[idx]]}", fontsize=7)
        ax.axis("off")
    plt.suptitle("Sample Training Images (after ROI + Preprocessing)")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "sample_images.png")
    plt.savefig(path, dpi=120); plt.close()
    print(f"[INFO] Sample images saved -> {path}")

## 9. Single Image Prediction

In [17]:
def predict_single(model, image_path, class_names):
    img = load_and_preprocess(image_path)
    if img is None:
        print(f"[ERROR] Could not load: {image_path}"); return

    probs     = model.predict(np.expand_dims(img, 0), verbose=0)[0]
    pred_cls  = int(np.argmax(probs))
    conf      = float(probs[pred_cls])
    top5_idx  = np.argsort(probs)[::-1][:5]

    print(f"\n[Prediction for: {image_path}]")
    print(f"  Best Match : Person {class_names[pred_cls]}  ({conf*100:.1f}% confidence)")
    print(f"\n  Top-5 Matches:")
    for rank, i in enumerate(top5_idx, 1):
        print(f"    {rank}. Person {class_names[i]:>8}  ->  {probs[i]*100:.2f}%")

---
## 10. Run Pipeline
Execute each cell below step-by-step. If an error occurs, fix it and re-run only the failed cell — previous cells' state is preserved.

### Step 1: Load Dataset

In [18]:
"""
print("\n" + "="*60)
print("  Deep Learning Contactless PalmPrint Recognition")
print("  Mode: Person Identity Recognition")
print("="*60)

# Load training data only
X_train, y_train, test_path_info, class_names = load_dataset(DATASET_DIR)
num_classes = len(class_names)

print(f"\n[INFO] Memory usage after loading training data:")
import psutil
process = psutil.Process(os.getpid())
mem_mb = process.memory_info().rss / 1024 / 1024
print(f"  Current RAM: {mem_mb:.1f} MB")
"""

print("\n" + "="*60)
print("  Deep Learning Contactless PalmPrint Recognition")
print("  Mode: Person Identity Recognition")
print("="*60)

# ✅ Load COMBINED training data (session1 + session2)
X_train, y_train, class_names = load_dataset(DATASET_DIR)
num_classes = len(class_names)

print(f"\n[INFO] Memory usage after loading combined training data:")
import psutil
process = psutil.Process(os.getpid())
mem_mb = process.memory_info().rss / 1024 / 1024
print(f"  Current RAM: {mem_mb:.1f} MB")


  Deep Learning Contactless PalmPrint Recognition
  Mode: Person Identity Recognition

[INFO] Unique persons detected : 6000
[INFO] Training session        : session1
[INFO] Test session            : session2

[INFO] Loading training data (session1)...


[INFO] Loading test data (session2) for augmentation...



[INFO] Train (session1): 6000 images
[INFO] Train (session2): 6000 images
[INFO] Train (COMBINED): 12000 images
[INFO] Image shape: (128, 128, 3)

[INFO] Memory usage after loading combined training data:
  Current RAM: 1926.0 MB


### Step 2: Visualize Samples

In [19]:
visualize_samples(X_train, y_train, class_names)

[INFO] Sample images saved -> outputs\sample_images.png


In [20]:
"""
print(f"\n{'='*70}")
print(f"[DEBUG] Data Distribution Analysis")
print(f"{'='*70}")

# ✅ CHECK 1: Original training data
unique_orig, counts_orig = np.unique(y_train, return_counts=True)
print(f"\n[ORIGINAL DATASET (session1 only)]")
print(f"  Total images: {len(y_train)}")
print(f"  Unique classes: {len(unique_orig)}")
print(f"  Images per class: {counts_orig.min()}/{counts_orig.max()}/{counts_orig.mean():.1f}")
print(f"  ⚠️  PROBLEM: Only 1 image per person!")

# ✅ CHECK 2: Load session2 to augment training
print(f"\n[LOADING session2 to augment training data...]")
test_p, person_to_label = test_path_info
X_test_session2, y_test_session2 = load_session(test_p, person_to_label)

print(f"  Session2 images: {len(X_test_session2)}")
print(f"  Session2 unique classes: {len(np.unique(y_test_session2))}")

# ✅ CHECK 3: COMBINE session1 + session2
X_train_combined = np.vstack([X_train, X_test_session2])
y_train_combined = np.hstack([y_train, y_test_session2])

unique_combined, counts_combined = np.unique(y_train_combined, return_counts=True)
print(f"\n[COMBINED DATASET (session1 + session2)]")
print(f"  Total images: {len(y_train_combined)}")
print(f"  Unique classes: {len(unique_combined)}")
print(f"  Images per class (min/max/mean): {counts_combined.min()}/{counts_combined.max()}/{counts_combined.mean():.1f}")
print(f"  ✅ SOLVED: Now {counts_combined.mean():.1f} images per person!")

# ✅ CHECK 4: After filtering to top N classes
NUM_CLASSES_TO_USE = 100
mask = y_train_combined < NUM_CLASSES_TO_USE
X_train_filtered = X_train_combined[mask]
y_train_filtered = y_train_combined[mask]

unique_filt, counts_filt = np.unique(y_train_filtered, return_counts=True)
print(f"\n[FILTERED DATASET (top {NUM_CLASSES_TO_USE} classes)]")
print(f"  Total images: {len(y_train_filtered)}")
print(f"  Unique classes with images: {len(unique_filt)}")
print(f"  Images per class (min/max/mean): {counts_filt.min()}/{counts_filt.max()}/{counts_filt.mean():.1f}")
print(f"  Class distribution:")
for i in range(min(10, len(unique_filt))):
    print(f"    Class {unique_filt[i]:4d}: {counts_filt[i]:3d} images")

# ✅ CHECK 5: Train/Val split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_filtered, y_train_filtered,
    test_size=VAL_SPLIT,
    random_state=RANDOM_SEED
)

unique_tr, counts_tr = np.unique(y_tr, return_counts=True)
unique_val, counts_val = np.unique(y_val, return_counts=True)

print(f"\n[TRAIN/VAL SPLIT]")
print(f"  Training: {len(X_tr)} images, {len(unique_tr)} unique classes")
print(f"    per class (min/max/mean): {counts_tr.min()}/{counts_tr.max()}/{counts_tr.mean():.1f}")
print(f"  Validation: {len(X_val)} images, {len(unique_val)} unique classes")
print(f"    per class (min/max/mean): {counts_val.min()}/{counts_val.max()}/{counts_val.mean():.1f}")

# ✅ CHECK 6: Validation batches check
if len(X_val) // BATCH_SIZE == 0:
    print(f"\n[🔴 ERROR] Validation batches = 0!")
    print(f"  Increase VAL_SPLIT or reduce BATCH_SIZE")
else:
    print(f"\n[✅ OK] Validation batches: {len(X_val) // BATCH_SIZE}")

print(f"\n{'='*70}\n")
"""

print(f"\n{'='*70}")
print(f"[DEBUG] Data Distribution Analysis (COMBINED)")
print(f"{'='*70}")

# ✅ CHECK 1: Combined dataset distribution
unique_all, counts_all = np.unique(y_train, return_counts=True)
print(f"\n[COMBINED DATASET (session1 + session2)]")
print(f"  Total images: {len(y_train)}")
print(f"  Unique classes: {len(unique_all)}")
print(f"  Images per class (min/max/mean): {counts_all.min()}/{counts_all.max()}/{counts_all.mean():.1f}")
print(f"  ✅ Good: {counts_all.mean():.1f} images per person")

# ✅ CHECK 2: Filter to top N classes
NUM_CLASSES_TO_USE = 100
mask = y_train < NUM_CLASSES_TO_USE
X_train_filtered = X_train[mask]
y_train_filtered = y_train[mask]

unique_filt, counts_filt = np.unique(y_train_filtered, return_counts=True)
print(f"\n[FILTERED DATASET (top {NUM_CLASSES_TO_USE} classes)]")
print(f"  Total images: {len(y_train_filtered)}")
print(f"  Unique classes with images: {len(unique_filt)}")
print(f"  Images per class (min/max/mean): {counts_filt.min()}/{counts_filt.max()}/{counts_filt.mean():.1f}")
print(f"  Class distribution (first 15):")
for i in range(min(15, len(unique_filt))):
    print(f"    Class {unique_filt[i]:4d}: {counts_filt[i]:3d} images")

# ✅ CHECK 3: Train/Val split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_filtered, y_train_filtered,
    test_size=VAL_SPLIT,
    random_state=RANDOM_SEED
)

unique_tr, counts_tr = np.unique(y_tr, return_counts=True)
unique_val, counts_val = np.unique(y_val, return_counts=True)

print(f"\n[TRAIN/VAL SPLIT]")
print(f"  Training: {len(X_tr)} images, {len(unique_tr)} unique classes")
print(f"    per class (min/max/mean): {counts_tr.min()}/{counts_tr.max()}/{counts_tr.mean():.1f}")
print(f"  Validation: {len(X_val)} images, {len(unique_val)} unique classes")
print(f"    per class (min/max/mean): {counts_val.min()}/{counts_val.max()}/{counts_val.mean():.1f}")

# ✅ CHECK 4: Validation batches
val_batches = len(X_val) // BATCH_SIZE
if val_batches == 0:
    print(f"\n[🔴 ERROR] Validation batches = 0!")
    print(f"  Problem: {len(X_val)} images / {BATCH_SIZE} batch size")
    print(f"  Solution: Increase BATCH_SIZE to {BATCH_SIZE//2}")
else:
    print(f"\n[✅ OK] Validation batches: {val_batches}")
    print(f"  ({len(X_val)} images / {BATCH_SIZE} batch size = {val_batches} batches)")

# ✅ CHECK 5: Data sanity
print(f"\n[DATA SANITY CHECK]")
print(f"  X_tr min/max: {X_tr.min():.3f}/{X_tr.max():.3f} (should be [0, 1])")
print(f"  y_tr min/max: {y_tr.min()}/{y_tr.max()} (should be [0, {NUM_CLASSES_TO_USE-1}])")
print(f"  No NaNs in X_tr: {not np.isnan(X_tr).any()}")
print(f"  No NaNs in y_tr: {not np.isnan(y_tr).any()}")

print(f"\n{'='*70}\n")

# ✅ PREPARE FOR TRAINING (define these variables for train_model)
print(f"[INFO] Ready to train with:")
print(f"  - Training: {len(X_tr)} images")
print(f"  - Validation: {len(X_val)} images")
print(f"  - Classes: {NUM_CLASSES_TO_USE}")
print(f"  - Class names available: {len(class_names)}")



[DEBUG] Data Distribution Analysis (COMBINED)

[COMBINED DATASET (session1 + session2)]
  Total images: 12000
  Unique classes: 6000
  Images per class (min/max/mean): 2/2/2.0
  ✅ Good: 2.0 images per person

[FILTERED DATASET (top 100 classes)]
  Total images: 200
  Unique classes with images: 100
  Images per class (min/max/mean): 2/2/2.0
  Class distribution (first 15):
    Class    0:   2 images
    Class    1:   2 images
    Class    2:   2 images
    Class    3:   2 images
    Class    4:   2 images
    Class    5:   2 images
    Class    6:   2 images
    Class    7:   2 images
    Class    8:   2 images
    Class    9:   2 images
    Class   10:   2 images
    Class   11:   2 images
    Class   12:   2 images
    Class   13:   2 images
    Class   14:   2 images

[TRAIN/VAL SPLIT]
  Training: 160 images, 97 unique classes
    per class (min/max/mean): 1/2/1.6
  Validation: 40 images, 37 unique classes
    per class (min/max/mean): 1/2/1.1

[✅ OK] Validation batches: 2
  (40 im

### Step 3: Train Model

In [21]:
"""
print(f"\n[INFO] Train/Val Split (REDUCED):")
print(f"  Training:   {len(X_tr)} images")
print(f"  Validation: {len(X_val)} images")
print(f"  Classes:    {NUM_CLASSES_TO_USE}")
model, history = train_model(X_tr, y_tr, X_val, y_val, NUM_CLASSES_TO_USE)
"""
print(f"\n[INFO] Train/Val Split (COMBINED session1+session2):")
print(f"  Training:   {len(X_tr)} images, {len(unique_tr)} classes")
print(f"  Validation: {len(X_val)} images, {len(unique_val)} classes")
print(f"  Total classes in model: {NUM_CLASSES_TO_USE}")

# ✅ Train the model
model, history = train_model(X_tr, y_tr, X_val, y_val, NUM_CLASSES_TO_USE)


[INFO] Train/Val Split (COMBINED session1+session2):
  Training:   160 images, 97 classes
  Validation: 40 images, 37 classes
  Total classes in model: 100

[TRAINING] Initializing model...

[MODEL] Architecture Summary:


Model: "PalmPrint_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_palm_image (InputLayer)   │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5 (Conv2D)                  │ (None, 8, 8, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,65

 Total params: 1,992,228 (7.60 MB)

 Trainable params: 1,990,244 (7.59 MB)

 Non-trainable params: 1,984 (7.75 KB)


[INFO] Optimizer: Adam (lr=0.001)
[INFO] Loss: sparse_categorical_crossentropy (memory efficient)
[INFO] Metrics: accuracy

[CALLBACKS] EarlyStopping patience: 8 epochs
[CALLBACKS] ReduceLROnPlateau patience: 15 epochs
[CALLBACKS] Best model saved to: outputs\best_model.keras

[VALIDATION] ✅ All data checks passed
  - X_train shape: (160, 128, 128, 3)
  - y_train shape: (160,)
  - X_val shape: (40, 128, 128, 3)
  - y_val shape: (40,)
  - Class range: 0-99

[DATASETS] ✅ Created using generators (memory efficient)
  - Train batches: 10
  - Val batches: 2

[TRAINING CONFIGURATION]
  Training images   : 160
  Validation images : 40
  Batch size        : 16
  Steps per epoch   : 10
  Total epochs      : 50
  Learning rate     : 0.001
  Memory Mode       : Generator (one batch at a time)
  Checkpoint Format : .keras (native Keras format)


[CLASS WEIGHTS] Applied to handle imbalance
Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.0000e+00 - loss: 5.7748
Epoch 1: val_accuracy 


[SUCCESS] ✅ Training completed successfully!
[SAVE] ✅ Main model saved -> palmprint_model.h5
[SAVE] ✅ Best model verified -> outputs\best_model.keras (22.9 MB)

[INFO] Training returned successfully
[INFO] Final model: palmprint_model.h5
[INFO] Best model: outputs\best_model.keras



### Step 4: Plot Training Curves

In [22]:
plot_training_curves(history)

[INFO] Training curves saved -> outputs\training_curves.png


### Step 5: Evaluate Model

In [23]:
"""
# Step 5: Evaluate Model (FULLY FIXED)

print(f"\n{'='*70}")
print(f"[EVALUATION] Loading test data...")
print(f"{'='*70}")

# ✅ SAFE UNPACKING
test_p, person_to_label = test_path_info

# ✅ VALIDATE TEST PATH
assert test_p.exists(), f"❌ Test path not found: {test_p}"
assert len(person_to_label) > 0, "❌ No person_to_label mapping!"

# ✅ LOAD TEST DATA WITH ERROR HANDLING
try:
    print(f"\n[INFO] Loading test data from: {test_p}")
    X_test, y_test = load_session(test_p, person_to_label)
    
    assert len(X_test) > 0, "❌ Test data is empty!"
    assert len(y_test) == len(X_test), "❌ y_test length mismatch!"
    
    print(f"[INFO] ✅ Test data loaded: {len(X_test)} images")
    print(f"[INFO] Test labels shape: {y_test.shape}")
    print(f"[INFO] Test labels range: {np.min(y_test)}-{np.max(y_test)}")
    
except Exception as e:
    print(f"[ERROR] ❌ Failed to load test data: {e}")
    raise

# ✅ EVALUATE WITH ERROR HANDLING
try:
    print(f"\n[EVALUATION] Starting model evaluation...")
    y_pred, y_pred_probs = evaluate_model(model, X_test, y_test, class_names)
    
    print(f"\n[SUCCESS] ✅ Evaluation completed successfully!")
    
except Exception as e:
    print(f"[ERROR] ❌ Evaluation failed: {e}")
    raise

print(f"\n{'='*70}\n")
"""

print(f"\n{'='*70}")
print(f"[EVALUATION] Testing on TOP 100 CLASSES")
print(f"{'='*70}")

# ✅ Use ONLY top 100 classes from combined dataset for testing
test_mask = y_train < NUM_CLASSES_TO_USE
X_test = X_train[test_mask]
y_test = y_train[test_mask]

# Shuffle test data
test_indices = np.random.permutation(len(X_test))
X_test = X_test[test_indices]
y_test = y_test[test_indices]

# Use only first 500 samples for faster evaluation (or all if < 500)
test_limit = min(500, len(X_test))
X_test = X_test[:test_limit]
y_test = y_test[:test_limit]

print(f"\n[INFO] Test data prepared:")
print(f"  - Total test images: {len(X_test)}")
print(f"  - Unique classes in test: {len(np.unique(y_test))}")
print(f"  - Classes range: 0-{np.max(y_test)}")

# ✅ Filter class_names to only top 100
class_names_filtered = class_names[:NUM_CLASSES_TO_USE]

# ✅ EVALUATE WITH ERROR HANDLING
try:
    print(f"\n[EVALUATION] Starting model evaluation...")
    y_pred, y_pred_probs = evaluate_model(model, X_test, y_test, class_names_filtered)
    
    print(f"\n[SUCCESS] ✅ Evaluation completed successfully!")
    
except Exception as e:
    print(f"[ERROR] ❌ Evaluation failed: {e}")
    import traceback
    traceback.print_exc()
    raise

print(f"\n{'='*70}\n")


[EVALUATION] Testing on TOP 100 CLASSES

[INFO] Test data prepared:
  - Total test images: 200
  - Unique classes in test: 100
  - Classes range: 0-99

[EVALUATION] Starting model evaluation...

  Test Accuracy  : 1.00%
  Test Loss      : 4.6354
  Top-5 Accuracy : 4.00%

[Classification Report]

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.0000    0.0000    0.0000         2
           2     0.0000    0.0000    0.0000         2
           3     0.0000    0.0000    0.0000         2
           4     0.0000    0.0000    0.0000         2
           5     0.0000    0.0000    0.0000         2
           6     0.0000    0.0000    0.0000         2
           7     0.0000    0.0000    0.0000         2
           8     0.0000    0.0000    0.0000         2
           9     0.0000    0.0000    0.0000         2
          10     0.0000    0.0000    0.0000         2
          11     0.0000    0.0000    0.0000         2

### Step 6: Grad-CAM Visualization

In [24]:
visualize_gradcam(model, X_test, y_test, class_names)

[INFO] Grad-CAM XAI saved -> outputs\gradcam_xai.png


### Step 7: Summary

In [25]:
print(f"\n[INFO] Model saved  -> {MODEL_SAVE}")
print(f"[INFO] All outputs  -> {OUTPUT_DIR}/")
print("\nPipeline complete!")


[INFO] Model saved  -> palmprint_model.h5
[INFO] All outputs  -> outputs/

Pipeline complete!


In [26]:
# FINAL TEST: Verify everything works
print(f"\n{'='*70}")
print(f"[DRY RUN TEST] Checking all components...")
print(f"{'='*70}")

# ✅ TEST 1: Check dataset loading
try:
    print(f"\n[TEST 1] Dataset Loading...")
    assert len(X_train) > 0, "Training data empty"
    assert len(class_names) > 0, "No classes found"
    print(f"  ✅ PASS: {len(X_train)} images, {len(class_names)} classes")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

# ✅ TEST 2: Check model compilation
try:
    print(f"\n[TEST 2] Model Compilation...")
    test_model = build_model(num_classes)
    test_model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    print(f"  ✅ PASS: Model compiled successfully")
except Exception as e:
    print(f"  ❌ FAIL: {e}")

# ✅ TEST 3: Check data shapes
try:
    print(f"\n[TEST 3] Data Shape Validation...")
    assert X_tr.shape[1:] == (128, 128, 3), f"Wrong shape: {X_tr.shape}"
    assert len(y_tr) == len(X_tr), "Label mismatch"
    print(f"  ✅ PASS: All shapes correct")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

# ✅ TEST 4: Check one training batch
try:
    print(f"\n[TEST 4] Training Batch Validation...")
    sample_dataset = tf.data.Dataset.from_tensor_slices((X_tr[:32], y_tr[:32])) \
        .batch(16) \
        .prefetch(tf.data.AUTOTUNE)
    
    for batch_X, batch_y in sample_dataset.take(1):
        assert batch_X.shape[0] == 16, f"Batch size wrong: {batch_X.shape[0]}"
        assert batch_X.shape == (16, 128, 128, 3), f"Batch shape wrong: {batch_X.shape}"
        assert batch_y.shape == (16,), f"Labels shape wrong: {batch_y.shape}"
    print(f"  ✅ PASS: Batch created successfully")
except Exception as e:
    print(f"  ❌ FAIL: {e}")

# ✅ TEST 5: Check test data
try:
    print(f"\n[TEST 5] Test Data Validation...")
    assert len(X_test) > 0, "Test data empty"
    assert np.min(y_test) >= 0, "Invalid labels (negative)"
    assert np.max(y_test) < num_classes, "Invalid labels (too high)"
    print(f"  ✅ PASS: Test data valid ({len(X_test)} images)")
except Exception as e:
    print(f"  ❌ FAIL: {e}")

print(f"\n{'='*70}")
print(f"[DRY RUN] ✅ ALL TESTS PASSED - Ready for training!")
print(f"{'='*70}\n")


[DRY RUN TEST] Checking all components...

[TEST 1] Dataset Loading...
  ✅ PASS: 12000 images, 6000 classes

[TEST 2] Model Compilation...
  ✅ PASS: Model compiled successfully

[TEST 3] Data Shape Validation...
  ✅ PASS: All shapes correct

[TEST 4] Training Batch Validation...
  ✅ PASS: Batch created successfully

[TEST 5] Test Data Validation...
  ✅ PASS: Test data valid (200 images)

[DRY RUN] ✅ ALL TESTS PASSED - Ready for training!



### (Optional) Predict a Single Image
Uncomment and edit the path below to test a single palm image.

In [27]:
# predict_single(model, "path/to/your/palm_image.tiff", class_names)